# PyOccam Simplified Demo
Essential workflow for OCCAM analysis in 5 steps

In [1]:
# Step 1: Setup
import pyoccam
from pathlib import Path

# Use included demo data or your own
pkg_dir = Path(pyoccam.__file__).parent
DATA_FILE = pkg_dir / "dementia05.txt"
# Or use your uploaded file:
# DATA_FILE = "dementia05_search_fullup.csv"

print(f"PyOccam version: {pyoccam.__version__}")
print(f"Data file: {DATA_FILE}")

PyOccam 0.1.2 loaded successfully
PyOccam 0.1.2 loaded. Type pyoccam.help() for usage.
PyOccam version: 0.1.2
Data file: D:\projects\occam\pyoccam\dementia05.txt


In [2]:
pyoccam.help()


PyOccam 0.1.2 - Quick Help

LOADING DATA (returns data object with .manager attribute):
  dementia = pyoccam.load_dementia()      # Load dementia dataset
  landslides = pyoccam.load_landslides()  # Load landslides dataset
  data = pyoccam.load_data("file.txt")    # Load any file

USING DATA OBJECTS:
  print(dementia.n_samples)               # Number of samples
  print(dementia.feature_names)           # Feature variable names
  print(dementia.target_name)             # Dependent variable name
  manager = dementia.manager              # Access the VBMManager
  
  # Convenience method on data object:
  best = dementia.quick_search()          # Run search on this data

STANDARD WORKFLOW:
  # 1. Load data
  data = pyoccam.load_dementia()
  
  # 2. Get the manager
  manager = data.manager
  
  # 3. Configure (Don't include ID or Model!)
  manager.set_report_variables("Level$I, h, ddf, dLR, Alpha, %dH(DV), dAIC, dBIC")
  
  # 4. Run search
  report = manager.generate_search_report("loopless

In [ ]:
# Step 2: Load Data
manager = pyoccam.VBMManager()
success = manager.init_from_command_line(["occam", str(DATA_FILE)])

if success:
    print(f"✓ Data loaded")
    print(f"  Sample size: {manager.get_sample_size()}")
    print(f"  Variables: {len(manager.get_variable_list())}")
    print(f"  Test data: {'Yes' if manager.has_test_data() else 'No'}")
else:
    print("ERROR: Could not load data")

In [ ]:
# Step 3: Run Search
SEARCH_TYPE = "loopless-up"  # or "full-up"
SEARCH_LEVELS = 7
SEARCH_WIDTH = 3

print(f"Running {SEARCH_TYPE} search...")
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=manager.has_test_data()
)

# Display search results
print("\n" + "="*60)
print(search_report[:2000])  # First 2000 chars
print("..." if len(search_report) > 2000 else "")
print(f"\nModels evaluated: {manager.get_search_model_count()}")

In [ ]:
# Step 4: Select Best Model
best_by_bic = manager.get_best_model_by_bic()
best_by_aic = manager.get_best_model_by_aic()

print("Best models:")
print(f"  BIC: {best_by_bic}")
print(f"  AIC: {best_by_aic}")

# Use BIC model for fit
selected_model = best_by_bic
if selected_model:
    stats = manager.get_model_statistics(selected_model)
    print(f"\nSelected: {selected_model}")
    print(f"  Information: {stats.information:.2%}")
    print(f"  BIC: {stats.bic:.2f}")
    print(f"  Alpha: {stats.alpha:.6f}")
    print(f"  % Correct: {stats.pct_correct_data:.1f}%")

In [ ]:
# Step 5: Generate Fit Report
if selected_model:
    manager.set_fit_classifier_target("0")  # Target state for confusion matrix
    
    fit_report = manager.generate_fit_report(
        model_name=selected_model,
        target_state="0"
    )
    
    # Display first part of fit report
    print("FIT REPORT")
    print("="*60)
    print(fit_report[:1500])  # First 1500 chars
    print("..." if len(fit_report) > 1500 else "")
    
    # Save full report
    output_file = f"fit_{selected_model.replace(':', '_')}.txt"
    with open(output_file, 'w') as f:
        f.write(fit_report)
    print(f"\nFull report saved to: {output_file}")